In [31]:
import kagglehub
import pandas as pd
import os
import ast
import html
import re
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk 
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt_tab to /home/relja/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/relja/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Ucitavanje sirovih podataka

In [32]:
path = kagglehub.dataset_download("shuyangli94/foodcom-recipes-with-search-terms-and-tags")

In [33]:
csv_path = os.path.join(path, 'recipes_w_search_terms.csv')
df = pd.read_csv(csv_path)

In [34]:
def get_cuisine(x):
    if not isinstance(x, str):
        return None
    x_lower = x.lower()
    if 'italian' in x_lower:
        return 'Italian'
    if 'indian' in x_lower:
        return 'Indian'
    return None

italian_indian_df = df[df['search_terms'].apply(lambda x: isinstance(x, str) and ('italian' in x.lower() or 'indian' in x.lower()))].copy()
italian_indian_df['cuisine'] = italian_indian_df['search_terms'].apply(get_cuisine)
final_df = italian_indian_df[['name', 'steps', 'cuisine']].reset_index(drop=True)
final_df['steps'] = final_df['steps'].apply(lambda x: ' '.join(ast.literal_eval(x)) if isinstance(x, str) else x)
final_df['name'] = final_df['name'].str.lower()
final_df['steps'] = final_df['steps'].str.lower()

In [35]:
italian_sample = final_df[final_df['cuisine'] == 'Italian'].sample(n=10000, random_state=42)
indian_all = final_df[final_df['cuisine'] == 'Indian']

final_df = pd.concat([italian_sample, indian_all], ignore_index=True)
final_df.to_csv('./data/recipes_raw.csv')

Osnovne provere

In [36]:
print('Number of null rows\n', final_df.isna().sum())
print('Number of duplicated rows', final_df.duplicated().sum())
df_no_dub = final_df.drop_duplicates().reset_index(drop=True)

Number of null rows
 name       0
steps      0
cuisine    0
dtype: int64
Number of duplicated rows 10


Čišćenje teksta (HTML entiteti, nevidljivi/kontrolni karakteri)

In [37]:
# zero-width space (200b), line/paragraph separator (2028/2029), BOM (feff), nbsp (a0)
_invisible = [0x200b, 0x2028, 0x2029, 0xfeff, 0xa0]
INVISIBLE_CHARS = re.compile('[' + ''.join(chr(c) for c in _invisible) + ']')
CONTROL_CHARS = re.compile('[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = html.unescape(text)  # &amp; -> &, &rsquo; -> ’, &eacute; -> é, ...
    text = unicodedata.normalize('NFKC', text)
    text = INVISIBLE_CHARS.sub(' ', text)
    text = CONTROL_CHARS.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean = df_no_dub.copy()
df_clean['name'] = df_clean['name'].apply(clean_text)
df_clean['steps'] = df_clean['steps'].apply(clean_text)

Tokenizacija

In [38]:
df_clean['name_tokens'] = df_clean['name'].apply(word_tokenize)
df_clean['steps_tokens'] = df_clean['steps'].apply(word_tokenize)
df_clean.to_csv('./data/recipes_tokenized.csv')

Uklanjanje stop reči

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

df_clean['name_tokens'] = df_clean['name_tokens'].apply(remove_stopwords)
# NOTE: not applied to steps_tokens here anymore — steps_tokens stays intact
# as the seq2seq (RNN_enc/RNN_dec) target. Stopwords are stripped later,
# from a copy, to build the separate BoW content-classifier feature.

In [ ]:
import string
def remove_punctuation(tokens):
    return [token for token in tokens if token not in string.punctuation]

df_clean["name_tokens"] = df_clean["name_tokens"].apply(remove_punctuation)
# steps_tokens is left alone here for the same reason as above — see remove_stopwords cell.

Statisticka analiza podataka

In [41]:
df_clean['number_of_tokens'] = df_clean['steps_tokens'].apply(len)
df_clean['number_of_tokens'].describe()

count    16546.000000
mean        77.481204
std         50.569092
min          1.000000
25%         45.000000
50%         67.000000
75%         97.000000
max        756.000000
Name: number_of_tokens, dtype: float64

In [42]:
max_len = int(df_clean['number_of_tokens'].quantile(0.95))
df_clean_no_outliers = df_clean[
    df_clean['number_of_tokens'].between(13, max_len - 1)
]
print(df_clean_no_outliers['cuisine'].value_counts())
df_clean_no_outliers.to_csv('./data/recipes_cleaned.csv')

cuisine
Italian    9245
Indian     6148
Name: count, dtype: int64


In [43]:
#======================================================#

In [44]:
#remove numbers and measurement units
MEASUREMENT_WORDS = {
    "cup", "cups",
    "tbsp", "tablespoon", "tablespoons",
    "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds",
    "qt", "quart", "quarts",
    "pint", "pints",
    "g", "kg", "mg",
    "ml", "l",
    "inch", "inches",
    "degree", "degrees",
    "minute", "minutes",
    "hour", "hours"
}

def remove_numbers_measurements(tokens):
    cleaned = []

    for token in tokens:
        token = token.lower()

        # remove pure numbers and fractions
        if re.fullmatch(r"[\d¼½¾⁄/.-]+", token):
            continue

        if token in MEASUREMENT_WORDS:
            continue

        cleaned.append(token)

    return cleaned

In [ ]:
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(remove_numbers_measurements)

In [46]:
#remove tokenization artifacts
ARTIFACTS = {
    "'s",
    "'re",
    "'ve",
    "'ll",
    "'d",
    "'m",
    "n't",
    "--"
}

def remove_artifacts(tokens):
    return [t for t in tokens if t not in ARTIFACTS]

In [47]:
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(remove_artifacts)

In [48]:
df_clean_no_outliers["steps_tokens"]

0        [saute, onions, garlic, heavy, iron, pot, oliv...
1        [cooking, potatoes, let, stand, 5, minutes, sl...
2        [large, skillet, saute, onion, oil, tender, ad...
3        [large, pot, medium-low, heat, combine, olive,...
4        [heat, olive, oil, skillet, peel, gently, pres...
                               ...                        
16541    [break, egg, bowl, separate, white, yolk, beat...
16542    [mix, ingredients, except, yogurt, lemon, 1, g...
16543    [remove, skin, chicken, pieces, cut, slits, le...
16544    [preheat, ove, 300., combine, coriander, brown...
16545    [mix, yogurt, water, make, little, less, reall...
Name: steps_tokens, Length: 15393, dtype: object

In [49]:
#normalize unicode fractions 

def normalize_unicode_fractions(tokens):
    replacements = {
        "½": "1/2",
        "¼": "1/4",
        "¾": "3/4",
        "⁄": "/"
    }

    normalized = []

    for token in tokens:
        for old, new in replacements.items():
            token = token.replace(old, new)
        normalized.append(token)

    return normalized

In [50]:
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(normalize_unicode_fractions)

In [51]:
#split hyphenated words
def split_hyphenated(tokens):
    output = []

    for token in tokens:
        output.extend(token.replace("-", " ").split())

    return output

In [52]:
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(split_hyphenated)

Grananje: BoW obeležja za sadržajni klasifikator (steps_tokens_bow)

steps_tokens ostaje netaknut kao seq2seq target za RNN_enc/RNN_dec.
steps_tokens_bow je kopija sa uklonjenim stop rečima i interpunkcijom, namenjena samo BoW/multi-task klasifikatoru.

In [ ]:
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens"].apply(remove_stopwords)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_punctuation)

In [53]:
df_clean_no_outliers.to_csv('./data/recipes_final.csv')